# ViT 실습: 이미지를 패치 토큰으로 바꾸고 self-attention 계산하기

patchify와 attention 행렬을 직접 계산합니다.


In [ ]:
# numpy는 패치와 attention 계산, matplotlib은 시각화에 사용합니다.
import numpy as np
import matplotlib.pyplot as plt

# 재현 가능한 실험을 위해 시드를 고정합니다.
np.random.seed(33)


In [ ]:
# 16x16 장난감 이미지에 서로 떨어진 밝은 블록 두 개를 둡니다.
image = np.zeros((16, 16))
image[2:6, 2:6] = 1.0
image[10:14, 10:14] = 0.8
image = np.clip(image + np.random.normal(0, 0.03, image.shape), 0, 1)
plt.imshow(image, cmap="gray")
plt.title("toy image")
plt.axis("off")
plt.show()


In [ ]:
# 이미지를 patch_size x patch_size 조각으로 나눕니다.
def patchify(x, patch_size=4):
    # 각 패치는 flatten되어 토큰 벡터가 됩니다.
    h, w = x.shape
    patches, positions = [], []
    for row in range(0, h, patch_size):
        for col in range(0, w, patch_size):
            patches.append(x[row:row+patch_size, col:col+patch_size].reshape(-1))
            positions.append((row // patch_size, col // patch_size))
    return np.array(patches), positions

patches, positions = patchify(image)
print("패치 개수:", len(patches))
print("패치 하나의 벡터 길이:", patches.shape[1])
print("처음 5개 패치 위치:", positions[:5])


In [ ]:
# 단순한 위치 인코딩을 만들어 패치 벡터에 붙입니다.
pos = np.array(positions, dtype=float)
pos = pos / pos.max()
tokens = np.concatenate([patches, pos], axis=1)
dim = tokens.shape[1]
attn_dim = 8
wq = np.random.normal(0, 0.2, (dim, attn_dim))
wk = np.random.normal(0, 0.2, (dim, attn_dim))
wv = np.random.normal(0, 0.2, (dim, attn_dim))

def softmax(x, axis=-1):
    # 수치 안정성을 위해 최댓값을 빼고 지수화를 합니다.
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=axis, keepdims=True)

def scaled_dot_product_attention(tokens):
    # 각 토큰을 query, key, value 벡터로 변환합니다.
    q = tokens @ wq
    k = tokens @ wk
    v = tokens @ wv
    scores = q @ k.T / np.sqrt(attn_dim)
    weights = softmax(scores, axis=1)
    output = weights @ v
    return output, weights

output, attention = scaled_dot_product_attention(tokens)
plt.imshow(attention, cmap="viridis")
plt.colorbar(label="attention weight")
plt.title("self-attention matrix")
plt.xlabel("key patch")
plt.ylabel("query patch")
plt.show()


In [ ]:
# 가장 밝은 패치가 어떤 패치를 많이 참고하는지 확인합니다.
patch_brightness = patches.mean(axis=1)
query_index = int(np.argmax(patch_brightness))
top_indices = np.argsort(attention[query_index])[::-1][:5]
print("가장 밝은 query patch 위치:", positions[query_index])
print("많이 참고한 patch 위치들:", [positions[i] for i in top_indices])
print("attention 값:", np.round(attention[query_index, top_indices], 3))


## 관찰 포인트

패치는 이미지의 단어 역할을 하고, self-attention은 멀리 떨어진 패치도 직접 연결할 수 있습니다.
